# Attribute Pipeline
Notebook này huấn luyện riêng attribute classifier với tên file và biến rõ ràng theo attribute pipeline.

In [ ]:
import os
import sys
import json
import random
from pathlib import Path

import torch
from torch.utils.data import DataLoader

current_dir = os.path.abspath(os.getcwd())
project_root = os.path.dirname(current_dir) if current_dir.endswith("notebooks") else current_dir
if project_root not in sys.path:
    sys.path.insert(0, project_root)
os.chdir(project_root)

from src.utils.config_loader import load_task_configs, print_config
from src.data.download import download_and_extract_metadata, download_vg_images, verify_download
from src.data.preprocessing import build_object_attribute_vocab_and_splits
from src.features.attribute_feature_extractor import extract_attribute_features
from src.data.attribute_dataset import build_attribute_datasets
from src.models.attribute import AttributeClassifier
from src.training.attribute_trainer import AttributeTrainer
from src.evaluation.metrics import compute_multilabel_metrics
from src.utils.memory import cleanup_cuda_memory

base_config = load_task_configs("configs/config.yaml")
attribute_config = load_task_configs("configs/config.yaml", "configs/attribute_config.yaml")

project_root_path = Path.cwd()
raw_dir = project_root_path / base_config.paths.raw_dir
image_dir = project_root_path / base_config.dataset.image_dir
processed_dir = project_root_path / attribute_config.dataset.processed_dir
feature_dir = processed_dir / attribute_config.dataset.feature_cache_dir

device = "cuda" if str(base_config.device).lower().startswith("cuda") and torch.cuda.is_available() else "cpu"

def normalize_input_mode(mode):
    mode = str(mode).lower().strip()
    aliases = {
        "grayscale": "gray",
        "grey": "gray",
        "edge": "contour",
        "edges": "contour",
    }
    mode = aliases.get(mode, mode)
    if mode not in {"rgb", "gray", "contour"}:
        raise ValueError("input mode phải là 'rgb', 'gray', hoặc 'contour'")
    return mode


def require_files(paths, label):
    missing = [str(path) for path in paths if not Path(path).exists()]
    if missing:
        raise FileNotFoundError(f"Thiếu {label}: {missing}")


def ensure_nonempty_cache(cache_path):
    cache_path = Path(cache_path)
    if not cache_path.exists():
        return False
    if cache_path.stat().st_size == 0:
        return False
    try:
        cache = torch.load(cache_path, map_location="cpu")
        return bool(cache)
    except Exception:
        return False


def cache_output(split_name, input_mode):
    suffix = None if input_mode == "rgb" else input_mode
    filename = f"{split_name}_features.pt" if suffix is None else f"{split_name}_{suffix}_features.pt"
    return feature_dir / filename


def collect_split_image_ids(processed_root, split_names):
    image_ids = set()
    for split_name in split_names:
        annotation_file = Path(processed_root) / split_name / "annotations.json"
        if not annotation_file.exists():
            raise FileNotFoundError(f"Thiếu annotation: {annotation_file}")

        with open(annotation_file, "r", encoding="utf-8") as f:
            raw = json.load(f)

        samples = raw if isinstance(raw, list) else raw.get("samples", [])
        image_ids.update(sample["image_id"] for sample in samples)

    return sorted(image_ids)


def ensure_images_for_splits(pipeline_name, processed_root, split_names):
    image_ids = collect_split_image_ids(processed_root, split_names)
    missing_ids = [img_id for img_id in image_ids if not (image_dir / f"{img_id}.jpg").exists()]

    print(f"[{pipeline_name}] Tổng ảnh tham chiếu: {len(image_ids)} | thiếu local: {len(missing_ids)}")
    if missing_ids:
        print(f"[{pipeline_name}] Đang tải bổ sung ảnh còn thiếu...")
        downloaded = download_vg_images(missing_ids, image_dir=str(image_dir))
        if len(downloaded) < len(missing_ids):
            print(f"[Warning] {pipeline_name}: còn {len(missing_ids) - len(downloaded)} ảnh chưa tải được.")


def evaluate_attribute_model(model, loader):
    model.eval()
    logits_list = []
    targets_list = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            features = batch.get("feature")
            if features is None:
                features = batch.get("attribute_feature")
            if features is None:
                features = batch.get("image")
            if features is None:
                features = batch.get("attribute_image")
            logits = model(features)
            logits_list.append(logits.cpu())
            targets_list.append(batch["attribute_labels"].cpu())

    return compute_multilabel_metrics(
        torch.cat(logits_list),
        torch.cat(targets_list),
        threshold=float(attribute_config.eval.attribute_threshold),
        threshold_mode=str(attribute_config.eval.attribute_threshold_mode),
        threshold_scale=float(attribute_config.eval.attribute_threshold_scale),
        threshold_min=float(attribute_config.eval.attribute_threshold_min),
        threshold_max=float(attribute_config.eval.attribute_threshold_max),
    )


download_data = bool(base_config.pipeline.download_data)
strict_sample_mode = bool(base_config.sampling.strict_mode)
sample_size = int(base_config.sampling.sample_size)
sample_seed = int(base_config.sampling.seed)
image_download_mode = "none" if strict_sample_mode else str(base_config.sampling.image_download_mode)
pre_extract_features = bool(base_config.pipeline.pre_extract_features)

feature_batch_size = int(base_config.feature_extraction.batch_size)
feature_resize_size = int(base_config.feature_extraction.resize_size)
feature_crop_size = int(base_config.feature_extraction.crop_size)
feature_mean = [float(x) for x in base_config.image.mean]
feature_std = [float(x) for x in base_config.image.std]
full_split_ratios = (
    float(base_config.split.train),
    float(base_config.split.val),
    float(base_config.split.test),
)
preprocess_split_ratios = (
    tuple(float(x) for x in base_config.sampling.split_ratios)
    if strict_sample_mode
    else full_split_ratios
)
max_samples = None if base_config.sampling.max_samples is None else int(base_config.sampling.max_samples)

attribute_learnable_backbone = bool(attribute_config.backbone.learnable_backbone)
attribute_use_cache = bool(attribute_config.dataset.use_feature_cache) and not attribute_learnable_backbone
attribute_input_mode = normalize_input_mode(attribute_config.dataset.attribute_input_mode)
attribute_batch_size = int(attribute_config.training.batch_size)
attribute_lr = float(attribute_config.training.lr)
attribute_max_epochs = int(attribute_config.training.max_epochs)
attribute_roi_size = int(base_config.image.roi_size)
attribute_hidden_dim = int(attribute_config.model.attribute_hidden_dim)
attribute_dropout = float(attribute_config.model.attribute_dropout)
attribute_num_layers = int(attribute_config.model.attribute_num_layers)

print_config(base_config)
print_config(attribute_config)
print(f"Project root: {project_root_path}")
print(f"Using device: {device}")
print("Pipeline: Attribute")
print(f"Strict sample mode: {strict_sample_mode}")
print(f"Sample size: {sample_size} | split ratios: {preprocess_split_ratios} | seed: {sample_seed}")
print(f"Processed root: {processed_dir}")
print(f"Feature extraction batch size: {feature_batch_size}")
print(f"Attribute learnable backbone: {attribute_learnable_backbone}")
print(f"Attribute input mode: {attribute_input_mode}")
print(f"Attribute batch size: {attribute_batch_size}, lr: {attribute_lr}, max_epochs: {attribute_max_epochs}")

CONFIGURATION
project_name: ImageCaptioningE2E
seed: 42
device: cuda
paths:
  data_dir: data
  raw_dir: data/raw
  processed_dir: data/processed
  feature_dir: data/features
  checkpoint_dir: checkpoints
  log_dir: logs
dataset:
  repo_id: ranjaykrishna/visual_genome
  cache_dir: data/raw/hf_cache
  image_dir: data/raw/images
image:
  roi_size: 224
  mean:
  - 0.485
  - 0.456
  - 0.406
  std:
  - 0.229
  - 0.224
  - 0.225
split:
  train: 0.7
  val: 0.15
  test: 0.15
training:
  batch_size: 1024
  num_workers: 2
  pin_memory: true
  max_epochs: 30
  early_stopping_patience: 5
  gradient_clip_val: 1.0
  log_every_n_steps: 0
optimizer:
  name: adamw
  lr: 0.0001
  weight_decay: 0.0001
scheduler:
  name: cosine
  warmup_epochs: 2
logging:
  use_tensorboard: true
  use_wandb: false
  wandb_project: vg-caption
pipeline:
  training_mode: all
  download_data: true
  pre_extract_features: true
sampling:
  strict_mode: true
  sample_size: 200
  split_ratios:
  - 0.8
  - 0.1
  - 0.1
  seed: 42
  

In [ ]:
# Load or verify raw dataif download_data:    print('Downloading metadata for the attribute pipeline...')    download_and_extract_metadata(raw_dir=str(raw_dir), keep_zip=False)    raw_status = verify_download(raw_dir=str(raw_dir))    missing_metadata = [name for name, ok in raw_status.items() if not ok]    if missing_metadata:        raise RuntimeError(f'Thiếu metadata sau khi tải: {missing_metadata}')else:    print('Bỏ qua tải mới, chỉ kiểm tra dữ liệu hiện có...')    raw_status = verify_download(raw_dir=str(raw_dir))    missing_metadata = [name for name, ok in raw_status.items() if not ok]    if missing_metadata:        raise RuntimeError(            'Thiếu dữ liệu RAW cần thiết: ' + ', '.join(missing_metadata) +            '. Hãy bật DOWNLOAD_DATA = True hoặc đặt đúng thư mục data/raw.'        )image_data_file = raw_dir / 'image_data.json'require_files([image_data_file], 'image_data.json')with open(image_data_file, 'r', encoding='utf-8') as f:    image_data = json.load(f)if strict_sample_mode:    all_image_ids = [img['image_id'] for img in image_data]    sample_count = min(sample_size, len(all_image_ids))    sample_image_ids = random.Random(sample_seed).sample(all_image_ids, sample_count)    print(f'Đã chọn trước {len(sample_image_ids)} image_id cho sample strict.')else:    sample_image_ids = None    print('Bỏ qua strict sample; sẽ dùng toàn bộ dữ liệu theo split mặc định.')if not strict_sample_mode and download_data and not attribute_use_cache and image_download_mode == 'sample':
    raise ValueError(
        "IMAGE_DOWNLOAD_MODE='sample' requires STRICT_SAMPLE_MODE=True when training from raw images; "
        "otherwise the notebook would download only a subset of images but still build the full split."
    )

if download_data and not attribute_use_cache:    if strict_sample_mode:        print(f'[Attribute] Đang tải {len(sample_image_ids)} ảnh sample cho raw-image training...')        downloaded_images = download_vg_images(sample_image_ids, image_dir=str(image_dir))        if len(downloaded_images) < len(sample_image_ids):            print(f'[Warning] Attribute: còn {len(sample_image_ids) - len(downloaded_images)} ảnh chưa tải được.')    elif image_download_mode == 'sample':        sample_ids = [img['image_id'] for img in image_data[:sample_size]]        print(f'[Attribute] Đang tải {len(sample_ids)} ảnh sample cho raw-image training...')        downloaded_images = download_vg_images(sample_ids, image_dir=str(image_dir))        if not downloaded_images:            raise RuntimeError('Không tải được ảnh mẫu nào.')    elif image_download_mode == 'all':        all_ids = [img['image_id'] for img in image_data]        print(f'[Attribute] Đang tải toàn bộ {len(all_ids)} ảnh cho raw-image training...')        downloaded_images = download_vg_images(all_ids, image_dir=str(image_dir))        if not downloaded_images:            raise RuntimeError('Không tải được ảnh nào.')    elif image_download_mode == 'none':        print('Bỏ qua tải ảnh theo cấu hình.')    else:        raise ValueError("IMAGE_DOWNLOAD_MODE phải là 'none', 'sample', hoặc 'all'.")    reduced_workers = 0 if device == 'cpu' else min(int(base_config.training.num_workers), 2)    if reduced_workers != int(base_config.training.num_workers):        print(f'[Attribute] Giảm num_workers từ {int(base_config.training.num_workers)} xuống {reduced_workers} cho raw-image mode.')    base_config.training.num_workers = reduced_workers    reduced_batch_size = min(attribute_batch_size, 8 if device == 'cpu' else 16)    if reduced_batch_size != attribute_batch_size:        print(f'[Attribute] Giảm batch size từ {attribute_batch_size} xuống {reduced_batch_size} cho raw-image mode.')    attribute_batch_size = reduced_batch_sizebuild_object_attribute_vocab_and_splits(    raw_dir=str(raw_dir),    processed_dir=str(processed_dir),    max_objects=int(attribute_config.labels.max_objects),    max_attributes=int(attribute_config.labels.max_attributes),    sample_image_ids=sample_image_ids,    split_by_image_id=strict_sample_mode,    split_ratios=preprocess_split_ratios,    seed=sample_seed,)require_files(    [        processed_dir / 'object_vocab.json',        processed_dir / 'attribute_vocab.json',        processed_dir / 'train' / 'annotations.json',        processed_dir / 'val' / 'annotations.json',        processed_dir / 'test' / 'annotations.json',    ],    'Attribute pipeline processed files',)if pre_extract_features and attribute_use_cache:    feature_dir.mkdir(parents=True, exist_ok=True)    ensure_images_for_splits('Attribute', processed_dir, ['train', 'val', 'test'])    print('Extracting attribute features...')    for split_name in ['train', 'val', 'test']:        attribute_output = cache_output(split_name, attribute_input_mode)        extract_attribute_features(            annotation_file=str(processed_dir / split_name / 'annotations.json'),            image_dir=str(image_dir),            output_file=str(attribute_output),            backbone=str(attribute_config.backbone.name),            pretrained=bool(attribute_config.backbone.pretrained),            batch_size=feature_batch_size,            device=device,            resize_size=feature_resize_size,            crop_size=feature_crop_size,            mean=feature_mean,            std=feature_std,            input_mode=attribute_input_mode,        )        if not ensure_nonempty_cache(attribute_output):            raise RuntimeError(f'Attribute feature cache rỗng: {attribute_output}')else:    print('Bỏ qua feature extraction theo cấu hình PRE_EXTRACT_FEATURES hoặc learnable backbone.')attribute_train_ds, attribute_val_ds, attribute_test_ds = build_attribute_datasets(    processed_dir=str(processed_dir),    image_dir=str(image_dir),    roi_size=attribute_roi_size,    use_feature_cache=attribute_use_cache,    feature_cache_dir=str(feature_dir),    input_mode=attribute_input_mode,    max_samples=max_samples,    train_horizontal_flip_p=float(attribute_config.augmentation.random_horizontal_flip),    train_color_jitter=bool(attribute_config.augmentation.color_jitter.enabled),    train_brightness=float(attribute_config.augmentation.color_jitter.brightness),    train_contrast=float(attribute_config.augmentation.color_jitter.contrast),    train_saturation=float(attribute_config.augmentation.color_jitter.saturation),    train_hue=float(attribute_config.augmentation.color_jitter.hue),    train_random_erasing_p=float(attribute_config.augmentation.random_erasing_p),    train_resize_delta=int(attribute_config.augmentation.resize_delta),    mean=feature_mean,    std=feature_std,)attribute_model = AttributeClassifier(    num_attributes=attribute_train_ds.num_attributes,    feature_dim=int(attribute_config.backbone.feature_dim),    hidden_dim=attribute_hidden_dim,    dropout=attribute_dropout,    num_layers=attribute_num_layers,    backbone_name=str(attribute_config.backbone.name) if attribute_learnable_backbone else None,    pretrained=bool(attribute_config.backbone.pretrained),    freeze_backbone=bool(attribute_config.backbone.freeze_backbone),    learnable_backbone=attribute_learnable_backbone,    device=device,)attribute_optimizer = torch.optim.AdamW(    attribute_model.parameters(),    lr=attribute_lr,    weight_decay=float(base_config.optimizer.weight_decay),)attribute_train_loader = DataLoader(    attribute_train_ds,    batch_size=attribute_batch_size,    shuffle=True,    num_workers=int(base_config.training.num_workers),    pin_memory=bool(base_config.training.pin_memory and device == 'cuda'),)attribute_val_loader = DataLoader(    attribute_val_ds,    batch_size=attribute_batch_size,    shuffle=False,    num_workers=int(base_config.training.num_workers),    pin_memory=bool(base_config.training.pin_memory and device == 'cuda'),)attribute_test_loader = DataLoader(    attribute_test_ds,    batch_size=attribute_batch_size,    shuffle=False,    num_workers=int(base_config.training.num_workers),    pin_memory=bool(base_config.training.pin_memory and device == 'cuda'),)attribute_trainer = AttributeTrainer(    model=attribute_model,    train_loader=attribute_train_loader,    val_loader=attribute_val_loader,    optimizer=attribute_optimizer,    use_auto_class_weights=True,    attribute_threshold_mode=str(attribute_config.eval.attribute_threshold_mode),    attribute_threshold=float(attribute_config.eval.attribute_threshold),    attribute_threshold_scale=float(attribute_config.eval.attribute_threshold_scale),    attribute_threshold_min=float(attribute_config.eval.attribute_threshold_min),    attribute_threshold_max=float(attribute_config.eval.attribute_threshold_max),    freeze_backbone=bool(attribute_config.backbone.freeze_backbone),    freeze_epochs=int(attribute_config.backbone.freeze_epochs),    max_epochs=attribute_max_epochs,    early_stopping_patience=int(base_config.training.early_stopping_patience),    gradient_clip_val=float(base_config.training.gradient_clip_val),    log_every_n_steps=int(base_config.training.log_every_n_steps),    device=device,    use_amp=(device == 'cuda'),)print('Bắt đầu training attribute pipeline...')attribute_val_metrics = attribute_trainer.train()attribute_best_checkpoint = attribute_trainer.checkpoint_manager.get_best_checkpoint_path()print('Attribute pipeline completed.')print(f'Best attribute checkpoint: {attribute_best_checkpoint}')attribute_model = attribute_trainer.attribute_modelattribute_test_metrics = evaluate_attribute_model(attribute_model, attribute_test_loader)print('Attribute test metrics:')print(attribute_test_metrics)

In [ ]:
cleanup_cuda_memory(note='Attribute pipeline notebook finished')